Things need to do:
1. python -m venv [venv name]
2. source [venv name]/bin/activate
3. pip install ipykernel
4. ipython kernel install --user --name=[venv name]
5. Change kernal manually in local

In [ ]:
# !pip install polars openpyxl msoffcrypto-tool pandas xlsx2csv openpyxl fastexcel


In [ ]:
import polars as pl
from pathlib import Path
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
# from openpyxl.utils.exceptions import InvalidFileException
import msoffcrypto
from io import BytesIO
# import numpy as np
import gc
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [ ]:
# Configure Polars display settings
pl.Config.set_tbl_cols(-1)  # Limit to 10 columns
pl.Config.set_tbl_rows(5)  # Limit to 20 rows
pl.Config.set_tbl_width_chars(200)  # Set the width of columns to 25 characters
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dtype_separator(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_cell_alignment("RIGHT")
pl.Config.float_precision=3

gc.collect()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:

# # Path to the CSV file
# file_path = Path("OneDrive_1_10-14-2024/Enriched_Data/2885 Amplify ABM ESE One Time Enrichment_ESE_6570272 23 Sep 2024_1.xlsx").resolve()
# password = ""

#  # Read the Excel file using pandas
# pandas_df = pd.read_excel(file_path,
#                         engine= 'openpyxl',
#                         sheet_name='Sheet1',  # Specify the sheet name
#                         # usecols=['col1', 'col2'],  # Read specific columns
#                         na_values=[None],  # Handle missing values
                        
#                         )

# # Convert the pandas DataFrame to a Polars DataFrame
# polars_df = pl.from_pandas(pandas_df)

# df = polars_df.with_columns([
#     pl.col(column).cast(pl.Utf8) for column in polars_df.columns
# ])
# # Display the DataFrame
# display(df)

In [ ]:
# Define the file path and the password
file_path = Path("OneDrive_1_10-14-2024/Enriched_Data/2885 Amplify ABM ESE One Time Enrichment_ESE_6570272 29 Oct 2024 - New Contacts Germany (with State).xlsx").resolve()
password = ""
sheet_nm = "data"

# Decrypt the file using msoffcrypto
decrypted = BytesIO()
with open(file_path, 'rb') as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=password)
    office_file.decrypt(decrypted)

# Load the decrypted file into a pandas DataFrame
decrypted.seek(0)  # Reset the pointer to the beginning of the BytesIO object

polars_df = pl.read_excel(decrypted, sheet_name=sheet_nm, engine='calamine')

df = polars_df
# # Changing all columns data type to "String"
# df = polars_df.with_columns([
#     pl.col(column).cast(pl.Utf8) for column in polars_df.columns
# ])

df = df.rename(lambda col_nm: col_nm.lower())
df = df.rename(lambda col_nm: col_nm.replace(' ', '_'))

# display(df)

display(df.select(pl.col('id')).count())

In [ ]:
# for col in df.columns:
#     print(col,':', df[col].dtype)

Optimising dataframe

In [ ]:
# def reduce_memory_usage_pl(df, name):
#     """ Reduce memory usage by polars dataframe {df} with name {name} by changing its data types.
#         Original pandas version of this function: https://www.kaggle.com/code/arjanso/reducing-dataframe-memory-size-by-65 """
#     print(f"Memory usage of dataframe {name} is {round(df.estimated_size('mb'), 2)} MB")
#     Numeric_Int_types = [pl.Int8,pl.Int16,pl.Int32,pl.Int64]
#     Numeric_Float_types = [pl.Float32,pl.Float64]    
#     for col in df.columns:
#         col_type = df[col].dtype
#         try:
#             c_min = df[col].min()
#             c_max = df[col].max()
#         except Exception as e:
#             pass

#         if col_type in Numeric_Int_types:
#             if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
#                 df = df.with_columns(df[col].cast(pl.Int8))
#             elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
#                 df = df.with_columns(df[col].cast(pl.Int16))
#             elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
#                 df = df.with_columns(df[col].cast(pl.Int32))
#             elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
#                 df = df.with_columns(df[col].cast(pl.Int64))
#         elif col_type in Numeric_Float_types:
#             if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
#                 df = df.with_columns(df[col].cast(pl.Float32))
#             else:
#                 pass
#         # elif col_type == pl.Utf8:
#         #     df = df.with_columns(df[col].cast(pl.Categorical))
#         else:
#             pass
#     print(f"Memory usage of dataframe {name} became {round(df.estimated_size('mb'), 2)} MB")
#     return df


# # usage example
# df = reduce_memory_usage_pl(df, "polar_dataframe")

In [ ]:
# for col in df.columns:
#     print(col,':', df[col].dtype)

In [ ]:
filter_ba_contact_status = df.filter(pl.col('status') =="Electronically Enriched")

display(filter_ba_contact_status.select(pl.col('id').count()))

### For Existing Contacts
Step 1: Verify if the contact is still with the same company provided initially

In [ ]:
filter_same_comapany = filter_ba_contact_status.filter(pl.col('adesk_contact_email_vs_contact_email_ba').is_null() ).unique()
display(filter_same_comapany.select(pl.col('id').count()))

In [ ]:
filter_Change_in_Email = filter_ba_contact_status.filter(pl.col('adesk_contact_email_vs_contact_email_ba')=="Change in Email" )
display(filter_Change_in_Email.select(pl.col('id').count()))

In [ ]:
filter_email_domain_vs_url_null = filter_Change_in_Email.filter(pl.col('email_domain_vs_url').is_null()).unique()
display(filter_email_domain_vs_url_null.select(pl.col('id').count()))

No calrity, so leaving this part for now

In [ ]:
# # need to work, don't know how to much extension of one string be part of another

# filter_email_domain_vs_url= filter_Change_in_Email.filter(pl.col('email_domain_vs_url')=='Different Domain')\
# .unique()
#     # .filter(pl.col('email_domain_cleaned').is_in(pl.col('url_cleaned')))\
#         # .unique()

# display(filter_email_domain_vs_url.select(pl.col('id').count()))


In [ ]:
list_upload_template_df = pl.concat([filter_same_comapany, filter_email_domain_vs_url_null ], how='vertical')
display(list_upload_template_df.select(pl.col('id').count()))


In [ ]:
# Fixing phone number
final_list_upload_template_df  = list_upload_template_df\
    .with_columns(pl.coalesce(pl.col(['contact_phone_ba','company_phone_ba'])).alias('contact_phone_ba'))
# Contact_Phone_BA, Contact_Mobile_BA, Company_Phone_BA
# Fixing product name 
final_list_upload_template_df = final_list_upload_template_df.with_columns(pl.coalesce(pl.col(['last_current_product_interest']), pl.lit('Dummy Product')).alias('last_current_product_interest'))

# Fixing contact country name 
final_list_upload_template_df = final_list_upload_template_df.with_columns(pl.coalesce(pl.col(['contact_country_ba', 'contact_country'])).alias('contact_country_ba'))

# Fixing email name 
final_list_upload_template_df = final_list_upload_template_df.with_columns(pl.coalesce(pl.col(['contact_email_ba', 'contact_email']).str.to_lowercase()).alias('Email') )

# Fixing city name 
final_list_upload_template_df = final_list_upload_template_df.with_columns(pl.coalesce(pl.col(['contact_city_ba', 'contact_city'])).alias('City') )

# Fixing Address name 
final_list_upload_template_df = final_list_upload_template_df.with_columns(
    pl.concat_str(
        [pl.col('contact_address1_ba'),pl.col('contact_address2_ba')], separator=' ', ignore_nulls=True
    ).alias('Address')
)


final_list_upload_template_df = final_list_upload_template_df.with_columns(
    pl.when(pl.col('contact_country_ba').str.contains('India')).then(pl.lit('IN'))
        .when(pl.col('contact_country_ba').str.contains('United States')).then(pl.lit('US'))
        .when(pl.col('contact_country_ba').str.contains('Germany')).then(pl.lit('DE'))
        .otherwise(pl.lit('No Match'))
        .alias('ISO_Code')
)

In [ ]:
###########################
# Temporary
###########################

final_list_upload_template_df = final_list_upload_template_df.select(
                                                                    pl.col('Email')
                                                                    ,pl.col('contact_first_name_ba').alias('First Name')
                                                                    ,pl.col('contact_last_name_ba').alias('Last Name')
                                                                    ,pl.col('contact_country_ba').alias('Country')
                                                                    ,pl.col('company_name_ba').alias('Company Name')
                                                                    ,pl.lit(None).alias('State')
                                                                    ,pl.col('contact_postcode_ba').alias('Postal Code')
                                                                    ,pl.col('contact_phone_ba').alias('Phone Number')
                                                                    ,pl.lit('6570272').alias('Activity Id')           # provided by Silvia
                                                                    ,pl.lit('6570272_Operational_Amplify ABM ESE One Time Enrichment').alias('Activity Name')         # need to discuss
                                                                    ,pl.lit('ADMIN').alias('Communication Channel')  # provided by Silvia
                                                                    ,pl.lit(None).alias('Lead Type')
                                                                    ,pl.lit('Non-WWM').alias('Program Name')    # need to discuss
                                                                    ,pl.col('last_current_product_interest').alias('Current Product Interest')
                                                                    ,pl.lit(None).alias('Lead Preference')           # need to discuss
                                                                    ,pl.lit(None).alias('Partner Account')  # need to discuss  
                                                                    ,pl.lit(None).alias('Response Type') # need to discuss
                                                                    ,pl.lit(None).alias('Qualification Notes') # need to discuss
                                                                    ,pl.lit(None).alias('Telenotes Private') # need to discuss
                                                                    ,pl.lit(None).alias('Lead Owner Email') # need to discuss
                                                                    ,pl.lit(None).alias('gdprPreferenceCategory') # need to discuss
                                                                    ,pl.lit(None).alias('gdprPreferenceValue') # need to discuss
                                                                    ,pl.lit(None).alias('gdprPreferenceSource') # need to discuss
                                                                    ,pl.col('contact_salutation_ba').alias('Salutation')
                                                                    ,pl.lit(None).alias('Middle Name')
                                                                    ,pl.col('contact_mobile_ba').alias('Mobile Phone Number')
                                                                    ,pl.col('contact_job_title_ba').alias('Job Title')
                                                                    ,pl.col('department_ba').alias('Department')
                                                                    ,pl.col('Address').alias('Address')
                                                                    ,pl.col('City').alias('City')
                                                                    ,pl.col('ISO_Code').alias('Country ISO Code')\
                                                                    ,pl.col('audience_role_e').alias('Audience Role E')
                                                                    ,pl.col('executive_level').alias('Executive Level')
                                                                    ,pl.col('adsk_industrysegment').alias('Industry Segment')  # need to discuss
                                                                    ,pl.col('adsk_industrysubsegment').alias('Industry Sub-Segment') # need to discuss
                                                                    ,pl.lit(None).alias('Profession') # need to discuss
                                                                    ,pl.lit(None).alias('Business Type') # need to discuss
                                                                    ,pl.lit(None).alias('Preferred Language of Communication') # need to discuss
                                                                    ,pl.lit(None).alias('Estimated Licenses') # need to discuss
                                                                    ,pl.lit(None).alias('Purchase Timeframe') # need to discuss
                                                                    ,pl.lit(None).alias('Lead Value') # need to discuss
                                                                    ,pl.lit(None).alias('Lead Currency') # need to discuss
                                                                    ,pl.lit(None).alias('Territory Skill') # need to discuss
                                                                    ,pl.col('marketo_lead_id').alias('marketo_id')  # verify
                                                                    ,pl.col('contact_uuid').alias('Contact UUID')    # verify
                                                                    ,pl.lit(None).alias('company UUID') # need to discuss
                                                                    ,pl.lit('Business Advantage').alias('Enrichment Vendor')
                                                                    ,pl.col('account_csn').alias('Account CSN')
                                                                    ,pl.col('jobfunction_ba').alias('Job Function')
                                                                )

final_list_upload_template_df = final_list_upload_template_df.unique()
display(final_list_upload_template_df.select(pl.all()) )
display(final_list_upload_template_df.select(pl.col('Email')).count() )

Write to csv file

In [ ]:
# final_list_upload_template_df.write_csv('final_list_upload_all.csv')

Write to Snowflake

In [ ]:
# Convert Polars DataFrame to Pandas DataFrame
pandas_list_upload_df = final_list_upload_template_df.to_pandas()

# Snowflake connection details
account = 'tata.us-east-1'
user = ''
password = ''
database = 'DEV'
schema = 'DIBAKAR'
warehouse = 'BSM_QA_WAREHOUSE_MEDIUM'
role = 'BSM_ABMAMPLIFY_SECURED'

# Establish connection
ctx = snowflake.connector.connect(
            account=account,
            user=user,
            password=password,
            warehouse=warehouse,
            DATABASE=database,   
            SCHEMA=schema,      
            role=role)

table_name = 'list_upload'

# Truncate the table if it exists
truncate_query = f""" TRUNCATE TABLE IF EXISTS {database}.{schema}.{table_name} """

cursr = ctx.cursor()

try:
    # Execute the TRUNCATE TABLE command
    cursr.execute(truncate_query)
    print(f"Table {table_name} DROP successfully.")
except Exception as e:
    print(f"Error DROP table {table_name}: {e}")
    exit()



# Write the DataFrame to Snowflake
success, nchunks, nrows, _ = write_pandas(conn=ctx, df=pandas_list_upload_df, table_name=table_name, database=database, schema=schema, overwrite=True, auto_create_table=True)

# Close the connection
ctx.close()

# Check if the upload was successful
if success:
    print(f"Successfully uploaded {nrows} rows in {nchunks} chunks.")
else:
    print("Failed to upload data.")

In [ ]:
# for cl in df.columns:
#     if 'account' in cl:
#         print(cl)

## Test

test for Email Domain vs URL blank whether url_cleaned != email_domain_cleaned

In [ ]:
# same_email_or_not = filter_same_comapany.filter(pl.col('contact_email').str.to_lowercase() != pl.col('contact_email_ba').str.to_lowercase())

# display(
#     same_email_or_not.select(pl.col(\
#         ["account_url","company_url_ba","email_domain_vs_url","url_cleaned", "contact_email","email_mao_usability"\
#             ,"owner_email","opted_in_to_emails_c","duplicateemail","emailvalid","contact_email_ba","email_domain_cleaned","adesk_contact_email_vs_contact_email_ba","last_current_product_interest"]
#     )
# )
#     )

# display(same_email_or_not.select(pl.col('id').count()))

In [ ]:
# change_in_email = filter_Change_in_Email.filter(pl.col('contact_email').str.to_lowercase() == pl.col('contact_email_ba').str.to_lowercase())
# display(
#     change_in_email.select(pl.col(\
#         ["account_url","company_url_ba","email_domain_vs_url","url_cleaned", "contact_email","email_mao_usability"\
#             ,"owner_email","opted_in_to_emails_c","duplicateemail","emailvalid","contact_email_ba","email_domain_cleaned","adesk_contact_email_vs_contact_email_ba"]
#     )
# )
#     )

In [ ]:
# email_domain_vs_url_null = filter_email_domain_vs_url_null.filter(pl.col('url_cleaned') != pl.col('email_domain_cleaned'))
# display(
#     email_domain_vs_url_null.select(pl.col(\
#         ["account_url","company_url_ba","email_domain_vs_url","url_cleaned", "contact_email","email_mao_usability"\
#             ,"owner_email","opted_in_to_emails_c","duplicateemail","emailvalid","contact_email_ba","email_domain_cleaned","adesk_contact_email_vs_contact_email_ba"]
#     )
# )
#     )

In [ ]:
# # Expected: url_cleaned != email_domain_cleaned
# email_domain_vs_url_diff_domain = filter_email_domain_vs_url.filter(pl.col('url_cleaned') == pl.col('email_domain_cleaned'))
# display(
#     email_domain_vs_url_diff_domain.select(pl.col(\
#         ["account_url","company_url_ba","email_domain_vs_url","url_cleaned", "contact_email","email_mao_usability"\
#             ,"owner_email","opted_in_to_emails_c","duplicateemail","emailvalid","contact_email_ba","email_domain_cleaned","adesk_contact_email_vs_contact_email_ba"]
#     )
# )
#     )